# آموزش full-archive روی Colab — patch store (370 مزرعه / 799 سایکل معتبر)

همان ۶ آموزش دست‌نخوردهٔ handover، روی GPU کولب: `patch_pixelwise` و `patch_cnn_gru`
× seeds `13, 42, 2026` — ۱۰ epoch.

**قبل از اجرا:**
1. Runtime → Change runtime type → **T4 GPU**
2. ریپو باید public باشد: `Mahdikalantari555/thesis_implementation`
3. فایل `thesis_data_full.zip` را در Drive خود در مسیر `MyDrive/thesis_colab/` بگذارید

`--resume` فعال است؛ اگر session قطع شد فقط سلول آموزش را دوباره اجرا کنید.
زمان تخمینی روی T4: حدود **۱٫۵ تا ۳ ساعت** برای هر ۶ آموزش (زمان دقیق در `run_manifest.json` ثبت می‌شود).

In [ ]:
import subprocess, sys, torch

assert torch.cuda.is_available(), (
    "GPU فعال نیست — Runtime > Change runtime type > T4 GPU را انتخاب کنید"
)
print("GPU:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "zarr==3.3.0", "lightning", "click"], check=True)
import zarr
print("zarr:", zarr.__version__)

In [ ]:
import subprocess

REPO_URL = "https://github.com/Mahdikalantari555/thesis_implementation.git"
r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL,
                    "/content/thesis_implementation"])
if r.returncode != 0:
    raise SystemExit("clone ناموفق — مطمئن شوید ریپو public شده است")
head = subprocess.run(["git", "-C", "/content/thesis_implementation", "rev-parse", "HEAD"],
                      capture_output=True, text=True).stdout.strip()
print("HEAD:", head)
print("*(handover ثبت‌شده روی commit 36b6161 است؛ HEAD جدید یعنی commitهای تازه‌تر)*")

In [ ]:
from google.colab import drive
import pathlib, shutil, zipfile

drive.mount("/content/drive")
ZIP = pathlib.Path("/content/drive/MyDrive/thesis_colab/thesis_data_full.zip")
assert ZIP.is_file(), f"زیپ دیتا پیدا نشد — آن را در {ZIP} بگذارید"
shutil.copy(ZIP, "/content/data.zip")
with zipfile.ZipFile("/content/data.zip") as z:
    z.extractall("/content/thesis_implementation/")
    print("extracted:", len(z.namelist()), "entries into repo data/")

In [ ]:
import os, sys, json

os.chdir("/content/thesis_implementation")
sys.path.insert(0, "/content/thesis_implementation/src")

from ssl_anomaly.monitor import splits as S

man = S.load_splits_locked(verify_cycle_files=False)
prov = json.loads(open("data/processed/patches.zarr/provenance.json").read())
assert prov["splits_manifest_sha256"] == man["manifest_sha256"], (
    "استور با تقسیم‌های قفل‌شده هم‌خوان نیست — بستهٔ دیتا را دوباره بسازید"
)
for name in S.SPLIT_NAMES:
    print(f"{name}: {len(man['cycles'].get(name, []))} cycles")
print("gates OK — store built from splits manifest",
      man["manifest_sha256"][:12])

In [ ]:
import os, subprocess, sys, time

os.chdir("/content/thesis_implementation")
env = dict(os.environ, PYTHONPATH="/content/thesis_implementation/src")
for arch in ["patch_pixelwise", "patch_cnn_gru"]:
    for seed in [13, 42, 2026]:
        rdir = f"checkpoints/patch/patch_full_ep10/{arch}_seed{seed}"
        extra = ["--resume"] if os.path.exists(f"{rdir}/last.pt") else []
        print(f"=== {arch} seed={seed}  start: {time.strftime('%H:%M:%S')} ===",
              flush=True)
        r = subprocess.run(
            [sys.executable, "-X", "utf8", "-u", "scripts/train_patch.py",
             "--arch", arch, "--seed", str(seed), "--epochs", "10",
             "--store", "data/processed/patches.zarr",
             "--norm", "data/processed/norm_stats_full.json",
             "--out", "checkpoints/patch/patch_full_ep10", *extra],
            env=env,
        )
        if r.returncode != 0:
            print(f"FAILED: {arch} seed={seed} (exit {r.returncode})", flush=True)
print(f"=== ALL DONE: {time.strftime('%H:%M:%S')} ===")

In [ ]:
import os, shutil

os.chdir("/content/thesis_implementation")
base = shutil.make_archive("/content/patch_full_ep10_results", "zip",
                           root_dir="checkpoints/patch", base_dir="patch_full_ep10")
dest = "/content/drive/MyDrive/thesis_colab/patch_full_ep10_colab_results.zip"
shutil.copy(base, dest)
print("results copied to Drive:", dest,
      f"({os.path.getsize(dest)/1e6:.1f} MB)")

## بعد از اتمام

* خروجی‌ها در Drive: `MyDrive/thesis_colab/patch_full_ep10_colab_results.zip`
  (۶ پوشهٔ run، هر کدام ۱۴ فایل: ۱۰ `epochNNN.pt` + `last.pt` + `norm_stats.json` + `progress.jsonl` + `run_manifest.json`)
* نکتهٔ reproducibility: seedها ثابت‌اند، ولی GPU در مقابل CPU ممکن است اختلاف float جزئی بدهد؛ دستگاه و wall time داخل `run_manifest.json` هر run ثبت شده است.
* گام بعد: zip را در `D:\RS\Thesis\thesis_implementation\checkpoints\patch\` از حالت zip خارج کنید تا sigma و eval ادامه پیدا کند.